# W08 tutorial — the Yield Curve Monitor

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the Yield Curve Monitor:** a one-page recession dashboard rebuilt from the FRED snapshot: the curve, the term spread, a probit, and one figure with a sentence underneath it that a desk could act on.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell the artefact exists and runs.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of what this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Nothing to install: this notebook uses only what Colab already ships
# (pandas, numpy, matplotlib, statsmodels, scipy, sklearn, pyarrow).

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

**FETCH gate 1: the monthly yields.** Everything starts from the monthly FRED file. It has one row per (date, series) pair, so we load it, `pivot` it into a table with one row per date and one column per series, and print what we actually got before trusting a single number in it.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

In [ ]:
fred = pd.read_parquet(DATA / "fred_monthly.parquet")

yields = fred.pivot(index="date", columns="series_id", values="value")
print(yields.columns.tolist())
print(yields.index.min().date(), "->", yields.index.max().date())

**Gate:** the columns include the CMT points (`DGS3MO`, `DGS1`, `DGS2`, `DGS5`, `DGS10`, `DGS30`) plus `T10Y3M` and others; dates are month-end; yields are in **per cent** (4.5 means 4.5% per year). Fifteen seconds of printing saves an afternoon of debugging.

### 1. FETCH gate 2: quarterly GDP and recessions

The outcomes live in the quarterly file: real GDP in levels, and the NBER recession flag. Same discipline — load it, pivot it, look at it.

In [ ]:
fredq = pd.read_parquet(DATA / "fred_quarterly.parquet")

q = fredq.pivot(index="date", columns="series_id", values="value")
q.index = q.index.to_period("Q")     # calendar quarter = the join key
print(q.tail(3))

**Gate:** two columns, `GDPC1` (real GDP, billions of chained dollars, in levels) and `USRECQ` (0/1 NBER flag). Converting the index to `Period("Q")` makes every later join unambiguous — we never rely on whether a quarterly date is stamped at the start or the end of the quarter.

### 2. TRANSFORM gate: the quarterly spread

The regressor is the 10y–3m spread, and it arrives monthly, so it has to move onto the same quarterly calendar as the outcomes. We keep each quarter's **quarter-end** value, grouping by `Period("Q")` so the index type converts at the same time.

In [ ]:
import statsmodels.api as sm

spread_m = yields["T10Y3M"].dropna()
ts_q = spread_m.groupby(spread_m.index.to_period("Q")).last()   # quarter-end value
ts_q.name = "spread"
print(f"n = {len(ts_q)}, {ts_q.index.min()} -> {ts_q.index.max()}")

**Gate:** the spread series starts in 1982, so expect roughly 170+ quarters. If you see 0 or a handful, your frequencies did not align — this is exactly the failure mode the gates exist for.

### 3. TRANSFORM gate: future growth, hand-checked

Annualised average growth over the **next** $k$ quarters (here $k=4$):

$$
Y_{t,t+4} = \frac{400}{4}\,\log\!\left(\frac{y_{t+4}}{y_t}\right)
$$

`shift(-4)` is what makes this *future* growth, and a sign error there is completely silent: the code still runs, it just answers a different question. So we also compute one value by hand from the raw GDP series and check that the two agree.

In [ ]:
y = q["GDPC1"]
growth4 = 400 * np.log(y.shift(-4) / y) / 4     # shift(-4) looks FORWARD
growth4.name = "k4"

t0 = pd.Period("2018Q4")
by_hand = 400 * np.log(y[t0 + 4] / y[t0]) / 4
print(f"By hand at {t0}: {by_hand:.4f}")
print(f"From the code:   {growth4[t0]:.4f}")

**Gate:** the two numbers match. The same hand-check at `2019Q4` is worth a moment — that window ends in `2020Q4`, so it spans the COVID collapse and rebound, and those quarters come back in section 5. The outcome is *allowed* to use the future — the regressor is not. That asymmetry is the whole look-ahead discipline.

`yields`, `q`, `ts_q` and the growth machinery are now in memory. Everything from here estimates on them.

---

### 4. Three snapshots of the curve

The curve is easiest to read as a picture. We take the six CMT points — `DGS3MO` at 0.25 years through `DGS30` at 30 years — and transpose them into a maturity-by-date table: maturity in years down the index, one column per date. Three dates carry the story: June 2021, June 2023, and whatever the last available date in the snapshot turns out to be. All three go onto one set of axes with maturity on the x-axis, and we read the 10y–3m and 10y–2y spreads off the final date.

In [ ]:
import matplotlib.pyplot as plt

mat_map = {"DGS3MO": 0.25, "DGS1": 1, "DGS2": 2,
           "DGS5": 5, "DGS10": 10, "DGS30": 30}
curve_data = yields[list(mat_map)].dropna()
last_date = curve_data.index.max()

dates = [pd.Timestamp("2021-06-30"), pd.Timestamp("2023-06-30"), last_date]
curves = curve_data.loc[dates].T                   # one column per date
curves.index = curves.index.map(mat_map)           # series_id -> maturity in years
curves.columns = curves.columns.strftime("%b %Y")

ax = curves.plot(marker="o", figsize=(8, 4.2), xticks=list(mat_map.values()))
ax.set_title("CMT yield curves from the unit snapshot")
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Yield (% p.a.)")
ax.legend(title="Date")
plt.show()

In [ ]:
snap = curve_data.loc[last_date]
print(f"As of {last_date.date()}:")
print(f"10y - 3m spread: {snap['DGS10'] - snap['DGS3MO']:.2f} pp")
print(f"10y - 2y spread: {snap['DGS10'] - snap['DGS2']:.2f} pp")

June 2021 is a textbook upward slope from near-zero short rates; June 2023 has the short end clearly *above* the long end — inversion, with the 10y–3m spread deeply negative. The latest curve depends on the snapshot's end date, so what carries the meaning is the sign of the two spreads printed above: positive is a normal slope, negative is the market pricing future cuts.

Two details worth flagging: the snapshot dates are **month-ends** (that is how `fred_monthly` is stamped, so `2021-06-30` exists in the index), and `dropna()` across all six maturities silently trims the table to start in September 1981, when the youngest of the six series (`DGS3MO`) begins — know your gaps before you select rows.

---

### 5. The growth regressions, across horizons

One horizon is an anecdote; a ladder of horizons is a result. We build annualised future growth for horizons $k \in \{1, 2, 3, 4, 6, 8\}$ quarters, join the quarterly spread onto it, and print the join gate — row count and date range — before estimating anything at all. Each horizon then gets its own regression of future growth on the spread with **Newey–West (HAC)** standard errors, `maxlags = max(k, 4)`, and the slope, HAC $t$-statistic and $n$ go into one table.

The second cell re-runs the eight-quarter horizon twice, once with HAC and once with plain OLS, so we can see what the correction is actually doing.

In [ ]:
horizons = [1, 2, 3, 4, 6, 8]
growth = pd.DataFrame(
    {f"k{k}": 400 * np.log(y.shift(-k) / y) / k for k in horizons}
)
joint = growth.join(ts_q, how="inner")
print(f"Join gate: n = {len(joint)}, {joint.index.min()} -> {joint.index.max()}")

rows = []
for k in horizons:
    df = joint[[f"k{k}", "spread"]].dropna()
    res = sm.OLS(df[f"k{k}"], sm.add_constant(df["spread"])).fit(
        cov_type="HAC", cov_kwds={"maxlags": max(k, 4)})
    rows.append({"horizon_q": k, "slope": res.params["spread"],
                 "t_hac": res.tvalues["spread"], "n": int(res.nobs)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
df8 = joint[["k8", "spread"]].dropna()
X8 = sm.add_constant(df8["spread"])
res_hac = sm.OLS(df8["k8"], X8).fit(cov_type="HAC", cov_kwds={"maxlags": 8})
res_ols = sm.OLS(df8["k8"], X8).fit()
print(f"k = 8 slope: {res_hac.params['spread']:.4f}")
print(f"t (HAC, 8 lags): {res_hac.tvalues['spread']:.2f}")
print(f"t (plain OLS):   {res_ols.tvalues['spread']:.2f}")

- Slopes are **positive** at business-cycle horizons ($k = 4$–$8$): a curve steeper by 1pp predicts annualised growth higher by roughly a quarter to a third of a percentage point over the next one to two years. HAC $t$-statistics are positive but modest on this modern sample — that is the honest replication result, not a bug (Notes 08, Exercise 3, checks whether COVID is the reason).
- Side by side at $k = 8$, the **plain** $t$-statistic is noticeably larger than the HAC one. At $k = 8$, consecutive outcomes share seven quarters of the same GDP data, so residuals are mechanically autocorrelated (MA(7) by construction — Notes 08 §5.3 has the derivation). Plain standard errors ignore that and come out too small, inflating $t$. The numbers "look better" precisely when they are wrong — which is why forgetting HAC is a trap rather than a shortcut.

---

### 6. The recession probit

Growth is continuous; "recession or not" is binary, which is what the probit is for. Following the paper, we estimate a probit of `USRECQ` on the spread lagged **four quarters** and print the coefficient table. Then — because a probit coefficient is not a probability — we ask `get_margeff()` for average marginal effects.

In [ ]:
probit_df = q[["USRECQ"]].join(ts_q.shift(4), how="inner").dropna()
X = sm.add_constant(probit_df["spread"])
probit_res = sm.Probit(probit_df["USRECQ"], X).fit(disp=0)
print(probit_res.summary().tables[1])

In [ ]:
print(probit_res.get_margeff().summary_frame())

The spread coefficient is clearly **negative** — around −0.4 to −1.0 in this snapshot, with $|z| > 3$ — and the average marginal effect is roughly −0.03 to −0.08.

- *Sign:* a steeper yield curve four quarters ago is associated with a lower probability of being in recession today — the expectations channel from the lecture.
- *Magnitude:* at the average observation, a 1pp steeper spread lowers recession probability by roughly 3–8 **percentage points** (the printed `dy/dx` is the exact figure for this snapshot).
- Reading that as −70pp would be wrong, because the coefficient acts on the *index inside* $\Phi$, not on the probability. The effect on probability is $\phi(x'\hat\beta)\hat\beta$: it depends on where you start, is largest near fitted $p = 0.5$, and is tiny in near-certain regimes. Only marginal effects are in probability units.

Note also the lag: `ts_q.shift(4)` (positive) pushes the regressor back in time, so today's outcome is explained by information available a year earlier. Same look-ahead discipline as lagging portfolio weights in Week 04.

---

### 7. The money chart and the current signal

This is the exhibit the whole week has been building towards: fitted recession probabilities over time with NBER recessions shaded behind them, then the last fitted probability in the sample and the spread reading that produced it.

A chart like this is only worth having if two or three honest sentences can sit underneath it — sentences that quote the fitted probability "as of" the snapshot's final quarter, own the 2022–24 false alarm, and stop short of promising a recession.

In [ ]:
import matplotlib.pyplot as plt

plot_df = probit_df.assign(prob=probit_res.predict(X))
plot_df.index = plot_df.index.to_timestamp()     # quarter -> its first day, for the date axis

fig, ax = plt.subplots(figsize=(8, 4.2))
for start in plot_df.loc[plot_df["USRECQ"] == 1].index:
    ax.axvspan(start, start + pd.DateOffset(months=3),
               color="#d62728", alpha=0.25, linewidth=0)
ax.plot(plot_df.index, plot_df["prob"], color="#0F6BBA", linewidth=1)
fig.suptitle("Recession probability from the 4-quarter-lagged term spread")
ax.set_title("Probit fitted values; NBER recessions shaded", fontsize=10)
ax.set_ylabel("Fitted probability")
plt.show()

In [ ]:
print(f"Latest quarter in snapshot: {plot_df.index[-1].date()}")
print(f"Spread 4 quarters earlier:  {probit_df['spread'].iloc[-1]:.2f} pp")
print(f"Fitted recession probability: {plot_df['prob'].iloc[-1]:.1%}")

The fitted probability spikes ahead of each shaded band (1990–91, 2001, 2008–09, 2020) — and then there is one enormous spike around 2024–25 with no shading after it.

Written out, with placeholders where the numbers move with the snapshot:

> *As of [final quarter], the term-structure model assigns a recession probability of [X]% at the one-year horizon, based on a 10y–3m spread of [Y]pp four quarters earlier. We treat this as a probability shifter rather than a forecast: the same model's strongest-ever signal, during the 2022–24 inversion, was not followed by an NBER recession within our sample — consistent with either a distorted post-QE term premium or simply a probabilistic signal behaving as designed. We therefore weight it alongside, not above, other indicators.*

What makes that paragraph usable: numbers quoted "as of" a stated date, an honest failure case disclosed, and no prophecy.

---

### 8. Verify the AI

A student asked an AI assistant to "improve the Week 08 recession model". It returned the code below, together with this claim:

> "I simplified the pipeline by removing the unnecessary lag. The refined model achieves a better in-sample fit, so I recommend it for forecasting recessions."

In [ ]:
# AI-generated "improvement" -- do not trust yet
df_ai = q[["USRECQ"]].join(ts_q, how="inner").dropna()
res_ai = sm.Probit(df_ai["USRECQ"],
                   sm.add_constant(df_ai["spread"])).fit(disp=0)
print(f"Pseudo R-squared: {res_ai.prsquared:.3f}")

The code runs without error and the claim sounds plausible, which is exactly what makes it dangerous. Three things need saying: what the flaw is, why "better in-sample fit" is no defence against it, and how the catch reads written down in two sentences — evidence and fix — so that it outlives the conversation.

- **The flaw: the spread is not lagged.** `ts_q` enters at quarter $t$ to "predict" recession at quarter $t$. That is *nowcasting*, not forecasting — the model answers "is the curve inverted while we are already in a recession?", which is useless in real time. The word "unnecessary" in the AI's explanation is doing a lot of unearned work: the lag *is* the forecast.
- Fitting both and comparing fitted probabilities through 2008 makes it concrete:

In [ ]:
df_ai = q[["USRECQ"]].join(ts_q, how="inner").dropna()
res_ai = sm.Probit(df_ai["USRECQ"],
                   sm.add_constant(df_ai["spread"])).fit(disp=0)

compare = pd.DataFrame(
    {"lagged_4q": [probit_res.params["spread"], probit_res.prsquared],
     "unlagged":  [res_ai.params["spread"],   res_ai.prsquared]},
    index=["spread_coef", "pseudo_R2"])
print(compare)

both = pd.DataFrame({
    "prob_lagged":   probit_res.predict(sm.add_constant(probit_df["spread"])),
    "prob_unlagged": res_ai.predict(sm.add_constant(df_ai["spread"])),
    "recession":     q["USRECQ"],
}).dropna()
print(both.loc[pd.Period("2007Q3"):pd.Period("2009Q2")].round(3))

The two models tell opposite stories around the crisis. In the table above, the lagged model — reading the curve that inverted in 2006–07 — assigns its highest recession risk in late 2007 and as 2008 begins, *before* the damage is visible. The unlagged model is at its calmest exactly then: it looks at the current curve, and its fitted probability only creeps up once the recession is under way and the curve has **re-steepened** because the Fed was slashing short rates *in response to* the crisis. Its spread coefficient even has the opposite sign — it has learned that a steep curve goes with already being in a recession, which describes the Fed's reaction rather than warning of anything. A model that is calm on the eve of 2008 is not an improvement.

In-sample fit compares how well each model explains quarters it has already seen; it says nothing about whether the model could have been *used* — the unlagged model needs quarter $t$'s spread to "predict" quarter $t$, information that arrives with the outcome. Whichever pseudo $R^2$ is higher, the two models answer different questions, and only the lagged one answers a forecasting question. This is Week 04's look-ahead villain in macro clothing.

The catch, in two sentences: *"The assistant removed the 4-quarter lag from the recession probit, converting a forecasting model into a nowcast, and justified it by in-sample fit. Evidence: going into the 2008 recession the unlagged model's fitted probability is at its lowest (about 7%) while the lagged model's is at its highest (about 20%), and the unlagged spread coefficient has the opposite sign; fix: restore `ts_q.shift(4)` and select the lag on theory, not fit."*

---

### 9. A second spread

The 10y–2y spread is the other headline inversion measure, so the obvious question is whether the result survives the switch. We build it from `yields` as `DGS10` − `DGS2`, collapse it to quarter-end frequency exactly as before, print the join gate, and re-estimate the recession probit with the same 4-quarter lag. What we are watching: the sample period, the coefficient, the pseudo $R^2$ — and whether that last one is comparable across the two models at all.

In [ ]:
spread2_m = (yields["DGS10"] - yields["DGS2"]).dropna()
ts2_q = spread2_m.groupby(spread2_m.index.to_period("Q")).last()
ts2_q.name = "spread2"

df2 = q[["USRECQ"]].join(ts2_q.shift(4), how="inner").dropna()
print(f"Join gate: n = {len(df2)}, {df2.index.min()} -> {df2.index.max()}")

res2 = sm.Probit(df2["USRECQ"], sm.add_constant(df2["spread2"])).fit(disp=0)
print(res2.summary().tables[1])
print(f"Pseudo R-squared: {res2.prsquared:.3f}")

The sample is longer — `DGS2` starts in 1976, six years before `T10Y3M` — so this model additionally sees the 1980 and 1981–82 recessions. The coefficient is again clearly **negative**: the qualitative story does not depend on which slope measure you pick.

The catch is the pseudo $R^2$: the two models are fitted on **different samples**, so their fit statistics are not directly comparable. A fair horse race re-estimates both on the common sample (1982 onwards) first — the same different-samples trap that once slipped into this unit's own Week 05 material, and a classic way to fool yourself in writing. Estimated on the aligned sample the comparison changes: in this snapshot the 10y–2y model's pseudo $R^2$ falls to roughly half the 10y–3m model's (about 0.07 against 0.14), even though the two coefficients are similar — the extra 1977–82 quarters had been flattering it.

#### ⚠️ Common mistakes this week
1. **Joining raw dates across frequencies.** Month-end timestamps joined to quarterly dates give an empty or silently misaligned frame. Convert both sides to `Period("Q")` first, and always print the join's row count and date range before proceeding.
2. **`shift(k)` vs `shift(-k)`.** Future growth needs `shift(-k)` on the outcome; the probit needs `shift(4)` on the regressor. If in doubt, hand-check one row against the raw GDP series, as in section 3.
3. **Reading probit coefficients as probability changes.** −0.7 is an index shift inside $\Phi$, not a 70pp fall in probability. Report `get_margeff()` for magnitudes.
4. **Skipping HAC on overlapping regressions.** Plain standard errors are too small by construction, so the $t$-statistics flatter you — exactly when you should trust them least. Use Newey–West with at least $k$ lags.
5. **Unit and sample drift.** The spread is in percentage points (1.5 means 1.5pp, not 0.015), and careless `dropna()` placement gives each horizon a different sample — the `n` column in your results table is where you catch it.